# DIA 6

In [ ]:
import xmlrpc.client
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 0)

# ===============================
# 1. Conexión con Odoo
# ===============================
ODOO_URL = "https://www.donssonusa.com"
db = "donsson-filters-florida-llc"
ODOO_USERNAME = "gbetancourt@donsson.com"
password = "DonssonFloridaCol"   

common = xmlrpc.client.ServerProxy(f"{ODOO_URL}/xmlrpc/2/common")
uid = common.authenticate(db, ODOO_USERNAME, password, {})
if not uid:
    print("Error de autenticación. Verifica las credenciales.")
    exit()

models = xmlrpc.client.ServerProxy(f"{ODOO_URL}/xmlrpc/2/object")

# ===============================
# 2. Extraer productos
# ===============================

def obtener_campos_modelo(model_name):
    fields = models.execute_kw(
        db, uid, password,
        model_name,
        "fields_get",
        [],
        {"attributes": ["string", "type", "relation", "required"]}
    )

    df = (
        pd.DataFrame.from_dict(fields, orient="index")
        .reset_index()
        .rename(columns={"index": "field_name"})
        .sort_values("field_name")
    )

    return df


In [ ]:
df_product_template_fields = obtener_campos_modelo("product.template")

#df_product_template_fields.to_csv("/home/jpcano/Donsson-Proyectos/DONSSON MIAMI/salidas/datos_product_template.csv")

In [ ]:
df_stock_quant_fields = obtener_campos_modelo("stock.quant")

#df_stock_quant_fields.to_csv("/home/jpcano/Donsson-Proyectos/DONSSON MIAMI/salidas/datos_stock_quant.csv")


In [ ]:
df_product_variant_fields = obtener_campos_modelo("product.product")


#df_product_variant_fields.to_csv("/home/jpcano/Donsson-Proyectos/DONSSON MIAMI/salidas/datos_product_varaint.csv")

In [ ]:
# PRODUCTOS
product_fields = [
    "id",
    "name",
    "default_code",
    "image_1920",
    "weight",#
    "volume",
    #"brand_id"  # no existe
]

products = models.execute_kw(
    db, uid, password,
    "product.template",
    "search_read",
    [[]],
    {"fields": product_fields}
)

df_products = pd.DataFrame(products)


## identificar sin foto ni medidas ni codigo

In [ ]:
df_sin_foto = df_products[
    df_products["image_1920"].isna() | (df_products["image_1920"] == False)
]

df_sin_codigo = df_products[
    df_products["default_code"].isna() | (df_products["default_code"] == "")
]

df_sin_logistica = df_products[
    (df_products["weight"].isna()) | (df_products["weight"] == 0) |
    (df_products["volume"].isna()) | (df_products["volume"] == 0)
]

def marca_sugerida(nombre):
    nombre = str(nombre).upper()
    if "DA" in nombre:
        return "Donsson"
    elif "BALDWIN" in nombre:
        return "Baldwin"
    else:
        return "Otro"

for df in [df_sin_foto, df_sin_codigo, df_sin_logistica]:
    df["Marca_Sugerida"] = df["name"].apply(marca_sugerida)
    df["Foto_Disponible_Colombia"] = ""





In [ ]:
with pd.ExcelWriter(
    "/home/jpcano/Donsson-Proyectos/DONSSON MIAMI/salidas/DIA_6_Diagnostico_Catalogo.xlsx",
    engine="xlsxwriter"
) as writer:
    df_sin_foto.to_excel(writer, sheet_name="Sin_Foto", index=False)
    df_sin_codigo.to_excel(writer, sheet_name="Sin_Internal_Reference", index=False)
    df_sin_logistica.to_excel(writer, sheet_name="Sin_Peso_Medidas", index=False)

In [ ]:
df_carga_codigo = df_sin_codigo[[
    "id",
    "name",
    "default_code"
]].copy()

df_carga_codigo.rename(columns={
    "default_code": "Internal_Reference (A completar)"
}, inplace=True)


In [ ]:
df_carga_marca = df_products[[
    "id",
    "name"
]].copy()

df_carga_marca["Marca (Donsson / Baldwin)"] = df_products.apply(marca_sugerida, axis=1)


In [ ]:
#with pd.ExcelWriter(
#    "DONSSON MIAMI/salidas/DIA_6_Plantillas_Carga.xlsx",
#    engine="xlsxwriter"
#) as writer:
#    df_carga_codigo.to_excel(writer, sheet_name="Carga_Internal_Reference", index=False)
#    df_carga_marca.to_excel(writer, sheet_name="Carga_Marca", index=False)


In [ ]:
import xmlrpc.client
import pandas as pd

# ===============================
# Conexión con Odoo 8
# ===============================
username = "juan.cano@donsson.com"
password = "1000285668"
url = "https://donsson.com"
db = "Donsson_produccion"

common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common")
uid = common.authenticate(db, username, password, {})
models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object")

products = models.execute_kw(
    db, uid, password,
    "product.template",
    "search_read",
    [[["image", "!=", False]]],
    {
        "fields": ["id", "name", "default_code", "image"],
        "limit": 5
    }
)

df_test = pd.DataFrame(products)

In [ ]:
import base64
import os

output_dir = "imagenes_odoo8_test"
os.makedirs(output_dir, exist_ok=True)

for _, row in df_test.iterrows():
    if row["image"]:
        image_data = base64.b64decode(row["image"])
        filename = f"{row['default_code'] or row['id']}.png"
        filepath = os.path.join(output_dir, filename)

        with open(filepath, "wb") as f:
            f.write(image_data)

        print(f"Imagen guardada: {filepath}")


# DIA 7

In [106]:
import pandas as pd
pesos_df = pd.read_csv("/home/jpcano/Donsson-Proyectos/DONSSON MIAMI/entradas/REPORTE SHIPSTATION SHIPMENTS COSTO DE DESPACHO-REFERENCIAS-PESOS Y MEDIDAS-no estan todos pero si muestra grande.csv")

pesos_df.head()

# Copia de seguridad
df_pesos = pesos_df.copy()

# Seleccionar solo columnas que nos sirven
df_pesos = df_pesos[[
    "Name",
    "Dimensions - Length",
    "Dimensions - Width",
    "Dimensions - Height",
    "Weight - WeightLbs"
]]

# Renombrar para que coincidan con Odoo
df_pesos = df_pesos.rename(columns={
    "Dimensions - Length": "x_studio_length",
    "Dimensions - Width": "x_studio_width",
    "Dimensions - Height": "x_studio_height"
})


In [107]:
import xmlrpc.client
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 0)

# ===============================
# 1. Conexión con Odoo
# ===============================
ODOO_URL = "https://www.donssonusa.com"
db = "donsson-filters-florida-llc"
ODOO_USERNAME = "gbetancourt@donsson.com"
password = "DonssonFloridaCol"   


common = xmlrpc.client.ServerProxy(f"{ODOO_URL}/xmlrpc/2/common")
uid = common.authenticate(db, ODOO_USERNAME, password, {})
if not uid:
    print("Error de autenticación. Verifica las credenciales.")
    exit()

models = xmlrpc.client.ServerProxy(f"{ODOO_URL}/xmlrpc/2/object")    
    
product_fields = [
    "id",
    "default_code",
    "name",
    "type",
    #"image_1920",
    "x_studio_length",   # AJUSTAR si el nombre real es otro
    "x_studio_width",   # AJUSTAR si el nombre real es otro
    "x_studio_height",
    "x_studio_weight_lb",
    #"volume",
    "x_studio_brand",
    "categ_id"

]

# ===============================
# DESCARGA DEL CATÁLOGO
# ===============================
products = models.execute_kw(
    db, uid, password,
    "product.template",
    "search_read",
    [[]],
    {
        "fields": product_fields,
        "limit": False
    }
)

df_catalogo = pd.DataFrame(products)

In [108]:
df_catalogo.sample(5)

,id,default_code,name,type,x_studio_length,x_studio_width,x_studio_height,x_studio_weight_lb,x_studio_brand,categ_id
1302,15862,False,BALDWIN BK6031,product,0.0,0.0,0.0,0.0,False,"[25, Filters / Other brands]"
5603,13095,False,BALDWIN PT566,product,0.0,0.0,0.0,0.0,False,"[25, Filters / Other brands]"
2743,17404,False,BALDWIN PA10208 KIT,product,0.0,0.0,0.0,0.0,False,"[25, Filters / Other brands]"
4154,14733,False,BALDWIN PA3637,product,0.0,0.0,0.0,0.0,False,"[25, Filters / Other brands]"
6613,4263,DA2523,DA2523 AIR FILTER,product,0.0,0.0,0.0,0.0,False,"[22, Filters / Donsson / Manufactured in Colombia]"


In [109]:
df_pesos.sample(5)

,Name,x_studio_length,x_studio_width,x_studio_height,Weight - WeightLbs
397,DA7808,13.0,13.0,17.0,8
369,DA2021,7.0,7.0,15.0,3
408,DA2946,NaN,NaN,NaN,0
86,DA2666,7.0,14.0,15.0,4
300,DA2688,11.0,11.0,18.0,8


In [110]:
def asignar_marca(nombre):
    nombre = str(nombre).upper()
    # Verifica si contiene "DA" o si comienza con "GS" o "GX"
    if "DA" in nombre or nombre.startswith("GS") or nombre.startswith("GX") or nombre.startswith("GA"):
        return "Donsson"
    elif "BALDWIN" in nombre:
        return "Baldwin"
    else:
        return ""

df_catalogo["x_studio_brand"] = df_catalogo["name"].apply(asignar_marca)


In [111]:
df_catalogo["x_studio_brand"].value_counts()

x_studio_brand
Baldwin    6468
Donsson     914
             16
Name: count, dtype: int64

In [112]:
#df_catalogo[df_catalogo["x_studio_brand"]==""]

In [113]:
df_donsson = df_catalogo.copy()

df_donsson["clave_producto"] = (
    df_donsson["name"]
    .astype(str)
    .str.strip()
    .str.split(" ")
    .str[0]
    .str.upper()
)



In [114]:
df_pesos = pesos_df[[
    "Name",
    "Dimensions - Length",
    "Dimensions - Width",
    "Dimensions - Height",
    "Weight - WeightLbs"
]].copy()

df_pesos = df_pesos.rename(columns={
    "Name": "clave_producto",
    "Dimensions - Length": "x_studio_length",
    "Dimensions - Width": "x_studio_width",
    "Dimensions - Height": "x_studio_height",
    "Weight - WeightLbs": "x_studio_weight_lb"
})


df_pesos["clave_producto"] = (
    df_pesos["clave_producto"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df_catalogo["clave_producto"] = (
    df_catalogo["default_code"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df_pesos = (
    df_pesos
    .dropna(subset=["x_studio_weight_lb"])
    .drop_duplicates(subset=["clave_producto"], keep="first")
    .reset_index(drop=True)
)



In [115]:
df_merge = df_donsson.merge(
    df_pesos,
    on="clave_producto",
    how="left"
)

df_merge.head()

,id,default_code,name,type,x_studio_length_x,x_studio_width_x,x_studio_height_x,x_studio_weight_lb_x,x_studio_brand,categ_id,clave_producto,x_studio_length_y,x_studio_width_y,x_studio_height_y,x_studio_weight_lb_y
0,6355,False,10079305-5001327,service,0.0,0.0,0.0,0.0,,"[3, All / Expenses]",10079305-5001327,NaN,NaN,NaN,NaN
1,6354,False,10079326-5000959 11-12,service,0.0,0.0,0.0,0.0,,"[3, All / Expenses]",10079326-5000959,NaN,NaN,NaN,NaN
2,25766,False,9205,product,0.0,0.0,0.0,0.0,,"[25, Filters / Other brands]",9205,NaN,NaN,NaN,NaN
3,25839,False,AIR FILTER DA8120: Replaces 20411815 20440934 49126 549126 AF26163M AF26472M,consu,0.0,0.0,0.0,0.0,Donsson,"[1, All]",AIR,NaN,NaN,NaN,NaN
4,12638,False,BALDWIN 10012,product,0.0,0.0,0.0,0.0,Baldwin,"[25, Filters / Other brands]",BALDWIN,NaN,NaN,NaN,NaN


In [116]:
for col in ["x_studio_weight_lb", "x_studio_length", "x_studio_width", "x_studio_height"]:
    df_merge[col] = df_merge[f"{col}_y"].combine_first(df_merge[f"{col}_x"])

df_merge = df_merge.drop(columns=[
    "x_studio_length_x",
    "x_studio_width_x",
    "x_studio_height_x",
    "x_studio_weight_lb_x",
    "x_studio_length_y",
    "x_studio_width_y",
    "x_studio_height_y",
    "x_studio_weight_lb_y"
])


In [117]:
con_peso = df_merge[df_merge["x_studio_height"]>0].count()
con_peso= con_peso[["name"]]

In [118]:
df_sin_peso = df_merge[df_merge["x_studio_weight_lb"]== 0]

df_sin_peso = df_sin_peso[["name"]]

#df_sin_peso.to_excel("/home/jpcano/Donsson-Proyectos/DONSSON MIAMI/salidas/productos_donsson_sin_pesos.xlsx")


In [119]:
total = len(df_merge)
sin_peso = total - con_peso

print(f"Total productos Donsson: {total}")
print(f"Con peso: {con_peso}")
print(f"Sin peso: {sin_peso}")


Total productos Donsson: 7398
Con peso: name    129
dtype: int64
Sin peso: name    7269
dtype: int64


In [120]:
ids = pd.read_csv("/home/jpcano/Donsson-Proyectos/DONSSON MIAMI/entradas/Product (product.template) (3).csv")
ids.head()

,id,name,standard_price,list_price,ebay_fixed_price,ebay_quantity,show_availability
0,__export__.product_template_25766_46858477,9205,63.12,1.00,0.0,1,True
1,__export__.product_template_25839_2fbde618,AIR FILTER DA8120: Replaces 20411815 20440934 49126 549126 AF26163M AF26472M,0.00,1.00,0.0,0,True
2,__export__.product_template_18629_5e996c84,BALDWIN 10012,14.06,17.58,0.0,1,True
3,__export__.product_template_12638_a52fbefb,BALDWIN 10012,14.06,17.58,0.0,1,True
4,__export__.product_template_12639_91216f4c,BALDWIN 10028,1.26,1.58,0.0,1,True


In [121]:
ids["name_norm"] = (
    ids["name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df_merge["name"] = (
    df_merge["name"]
    .astype(str)
    .str.strip()
    .str.upper()
)


In [122]:
df_upload = df_merge.merge(
    ids[["id", "name"]],
    on="name",
    how="inner"
)

df_upload_final = df_upload[[
    "id_y",
    "name",
    "x_studio_brand",
    "x_studio_weight_lb",
    "x_studio_length",
    "x_studio_width",
    "x_studio_height"
]].copy()


In [123]:
df_upload_final.head()

,id_y,name,x_studio_brand,x_studio_weight_lb,x_studio_length,x_studio_width,x_studio_height
0,__export__.product_template_25766_46858477,9205,,0.0,0.0,0.0,0.0
1,__export__.product_template_18629_5e996c84,BALDWIN 10012,Baldwin,0.0,0.0,0.0,0.0
2,__export__.product_template_12638_a52fbefb,BALDWIN 10012,Baldwin,0.0,0.0,0.0,0.0
3,__export__.product_template_18629_5e996c84,BALDWIN 10012,Baldwin,0.0,0.0,0.0,0.0
4,__export__.product_template_12638_a52fbefb,BALDWIN 10012,Baldwin,0.0,0.0,0.0,0.0


In [124]:
len(df_upload)


7826

In [126]:
df_upload_final.to_csv(
    "/home/jpcano/Donsson-Proyectos/DONSSON MIAMI/salidas/DIA_7_Carga_Peso_Volumen_Marca.csv",
    index=False
)
